In [ ]:
# divide the ndQTSA dataset into training and validation sets

import json
import random
import re
import os

input_file = "QTSA_new1.jsonl"
train_output = "SFT/Qwen3-0.6B/validation_QTSA_mix/ndQTSA_train.jsonl"
val_output = "SFT/Qwen3-0.6B/validation_QTSA_mix/ndQTSA_val.jsonl"

# Create directories if they don't exist
os.makedirs(os.path.dirname(train_output), exist_ok=True)
os.makedirs(os.path.dirname(val_output), exist_ok=True)

# Load all data
data = []
with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))

# Shuffle data
random.shuffle(data)

print(f"Total dataset size: {len(data)} pairs")

val_size = int(len(data) * 0.1)  
train_data = data[val_size:]     
val_data = data[:val_size]       

# Write training set (QTSA format)
with open(train_output, "w", encoding="utf-8") as f_train:
    for item in train_data:
        train_item = {
            "question": item.get("question", ""),
            "thinking": item.get("thinking", ""),
            "solution": item.get("solution", ""),
            "answer": item.get("answer", "")
        }
        f_train.write(json.dumps(train_item, ensure_ascii=False) + "\n")

# Write validation set (QTSA format)
with open(val_output, "w", encoding="utf-8") as f_val:
    for item in val_data:
        val_item = {
            "question": item.get("question", ""),
            "thinking": item.get("thinking", ""),
            "solution": item.get("solution", ""),
            "answer": item.get("answer", "")
        }
        f_val.write(json.dumps(val_item, ensure_ascii=False) + "\n")

print(f"Train dataset: {len(train_data)} pairs → {train_output}")
print(f"Validation dataset: {len(val_data)} pairs → {val_output}")

In [ ]:
# We conbine both mcQTSA and ndQTSA into mixed QSTA dataset (train & validation)

import json
import random
import os

ndQTSA_train_file = "SFT/Qwen3-0.6B/validation_QTSA_mix/ndQTSA_train.jsonl"
mcQTSA_train_file = "SFT/Qwen3-0.6B/validation_QTSA/mcQTSA_train_fixed.jsonl"
ndQTSA_val_file = "SFT/Qwen3-0.6B/validation_QTSA_mix/ndQTSA_val.jsonl"
mcQTSA_val_file = "SFT/Qwen3-0.6B/validation_QTSA/mcQTSA_val_fixed.jsonl"

train_output = "SFT/Qwen3-0.6B/validation_QTSA_mix/QTSA_train.jsonl"
val_output = "SFT/Qwen3-0.6B/validation_QTSA_mix/QTSA_val.jsonl"

os.makedirs(os.path.dirname(train_output), exist_ok=True)

def load_and_mix(file1, file2, output_file, dataset_type):
    data = []

    if os.path.exists(file1):
        with open(file1, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    data.append(json.loads(line))
        print(f"Loaded {len(data)} items from {file1}")
    else:
        print(f"Warning: File {file1} does not exist!")

    initial_count = len(data)
    if os.path.exists(file2):
        with open(file2, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    data.append(json.loads(line))
        print(f"Loaded {len(data) - initial_count} items from {file2}")
    else:
        print(f"Warning: File {file2} does not exist!")

    random.shuffle(data)
    print(f"Total {dataset_type} dataset size: {len(data)} pairs")

    with open(output_file, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    
    print(f"Mixed {dataset_type} dataset saved to: {output_file}")
    return len(data)

print("=== Processing Training Set ===")
train_count = load_and_mix(ndQTSA_train_file, mcQTSA_train_file, train_output, "training")

print("\n" + "="*50 + "\n")

print("=== Processing Validation Set ===")
val_count = load_and_mix(ndQTSA_val_file, mcQTSA_val_file, val_output, "validation")

print("\n" + "="*50)
print("=== Summary ===")
print(f"Final training set: {train_count} pairs")
print(f"Final validation set: {val_count} pairs")
print("Mixing completed successfully!")